In [20]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model(model="openai:gpt-4o")

In [21]:
class State(TypedDict):
    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str


class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: str


class IngredientsOutput(BaseModel):
    ingredients: List[Ingredient]

In [22]:
def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)
    response = structured_llm.invoke(
        f"List 5-8 infredients needed to make {state['dish']}"
    )
    return {"ingredients": response.ingredients}


def create_receipe(state: State):
    response = llm.invoke(
        f"Write a step by step cooking instruction for {state['dish']}, using these ingredients {state['ingredients']}"
    )
    return {"recipe_steps": response.content}


def describe_plating(state: State):
    response = llm.invoke(
        f"Describe how to beautifully plate this dish {state['dish']} based on this recipe {state['recipe_steps']}"
    )
    return {"plating_instructions": response.content}


def gate(state: State):
    ingredients = state["ingredients"]

    if len(ingredients) > 8 or len(ingredients) < 3:
        return False

    return True

In [25]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_receipe", create_receipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_conditional_edges(
    "list_ingredients",
    gate,
    {
        True: "create_receipe",
        False: END,
    },
)
graph_builder.add_edge("create_receipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [ ]:
graph.invoke({"dish": "미역국"})